# 09 · ResNet-50 y CLIP: EmoSet → anuncios (Positiva · Neutra · Negativa)

**Objetivo:** ver si aprender emociones en un dataset grande (EmoSet-118K, 8 emociones) mejora el reconocimiento de **valencia en los 2.000 anuncios** (3 clases, igual que en los notebooks 01 y 02).

**Etapa 1 · EmoSet (8 emociones de Mikels)**, con la partición oficial 80/5/15. EmoSet no tiene imágenes neutras: aquí se aprenden 4 emociones positivas y 4 negativas.

| Modelo | Qué se entrena | Referencia (Yang et al., ICCV 2023) |
|---|---|---|
| CLIP ViT-B/16 *zero-shot* | nada (solo texto: "a photo that evokes awe") | — |
| ResNet-50 congelada + sonda lineal | solo una capa lineal | — |
| CLIP congelado + sonda lineal | solo una capa lineal | — |
| **ResNet-50 ajustada entera** | toda la red | **74,0 %** (8 clases) · 93,5 % (2 valencias) |
| **CLIP ViT-B/16 ajustado entero** | toda la red | — (modelos CLIP ajustados reportan ~80 %) |

**Etapa 2 · anuncios (3 clases).** La clase **Neutra** (*Indiferencia*) solo existe en los anuncios, así que se aprende aquí. Se comparan 4 puntos de partida:
- ResNet-50 · ImageNet *vs* ResNet-50 · EmoSet (etapa 1)
- CLIP · original *vs* CLIP · EmoSet (etapa 1)

Con dos protocolos: **sonda lineal** (backbone congelado, como la sección 7b del notebook 01) y **ajuste completo** con 5 semillas. La partición y el test fijo son los de `pipeline.make_splits` (los mismos del notebook 01).

**Antes de ejecutar en Colab:** *Entorno de ejecución → Cambiar tipo de entorno → GPU*. Todo se guarda en `MyDrive/publicity_analysis/resultados/emoset_modelos/`. Si la sesión se corta, vuelva a ejecutar: el entrenamiento **se reanuda** desde la última época guardada y lo ya terminado no se repite.

**Tiempos orientativos** (dependen de la GPU y de los núcleos de CPU de la sesión):

| Paso | T4 | L4 / A100 |
|---|---|---|
| Preparar imágenes 256 px (solo la 1ª vez; luego se copia desde Drive en ~3 min) | 20–40 min | 10–20 min |
| Sondas lineales + *zero-shot* en EmoSet | ~15 min | ~5 min |
| ResNet-50, 10 épocas | 60–90 min | 20–40 min |
| CLIP ViT-B/16, 5 épocas | 50–80 min | 20–30 min |
| Etapa 2 (4 variantes × 2 protocolos × 5 semillas) | 30–45 min | 10–20 min |

## 1. Entorno: repositorio, Google Drive y GPU

In [ ]:
import os
import subprocess
import sys

REPO = 'publicity_analysis'
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # El repositorio trae pipeline.py, images_or/ y etiquetas_originales.csv (etapa 2)
    if not os.path.exists(f'/content/{REPO}'):
        subprocess.run(['git', 'clone', f'https://github.com/mlacarrasco/{REPO}.git', f'/content/{REPO}'], check=True)
    else:
        subprocess.run(['git', '-C', f'/content/{REPO}', 'pull'], check=True)
    os.chdir(f'/content/{REPO}')
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/publicity_analysis'
    WORK = '/content/work'                     # disco local de la VM (rápido)
else:
    import glob
    cands = [c for d in ('Mi unidad', 'My Drive')
             for c in glob.glob(os.path.expanduser(f'~/Library/CloudStorage/GoogleDrive-*/{d}/publicity_analysis'))]
    DRIVE_ROOT = next((c for c in cands if os.path.isdir(c)), '.')
    WORK = os.path.expanduser('~/datasets/work_09')
sys.path.insert(0, os.getcwd())

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    GPU_GB = props.total_memory / 1024**3
    print(f'GPU: {props.name} · {GPU_GB:.0f} GB · CPUs: {os.cpu_count()}')
else:
    GPU_GB = 0
    print('⚠️  Sin GPU. En Colab: Entorno de ejecución → Cambiar tipo de entorno → GPU.')

## 2. Configuración

`SMOKE = True` ejecuta todo con muestras pequeñas y 1 época (unos minutos), útil para comprobar que el notebook corre de principio a fin antes del entrenamiento real.

In [ ]:
import contextlib
import io
import json
import math
import multiprocessing as mp
import random
import shutil
import time
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from torchvision import models

import pipeline as pl
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()   # oculta el informe de carga de pesos de CLIP
pd.set_option('display.float_format', '{:.3f}'.format)

SMOKE = False

# Rutas
EMOSET_DIR = os.path.join(DRIVE_ROOT, 'emoset')
EMOSET_ZIP = os.path.join(EMOSET_DIR, 'EmoSet-118K.zip')
EMOSET_INDEX = os.path.join(EMOSET_DIR, 'emoset_index.csv')
CACHE_ZIP = os.path.join(EMOSET_DIR, 'EmoSet-118K_256.zip')   # imágenes reducidas, se crea la 1ª vez
IMG_DIR = os.path.join(WORK, 'emoset_256')
OUT = os.path.join(WORK, 'smoke_out') if SMOKE else os.path.join(DRIVE_ROOT, 'resultados', 'emoset_modelos')
for d in (WORK, IMG_DIR, OUT):
    os.makedirs(d, exist_ok=True)

# Hiperparámetros (lotes según la memoria de la GPU)
big = GPU_GB >= 30
HP = {
    'resnet50': {'epochs': 10, 'batch': 256 if big else 128, 'lr_backbone': 1e-4, 'lr_head': 1e-3, 'wd': 1e-4},
    'clip':     {'epochs': 5,  'batch': 256 if big else 64,  'lr_backbone': 1e-5, 'lr_head': 1e-3, 'wd': 0.05},
}
ADS = {'epochs': 15, 'patience': 5, 'batch': 32, 'seeds': (0, 1, 2, 3, 4),
       'lr_backbone': {'resnet50': 1e-4, 'clip': 1e-5}, 'lr_head': 1e-3}
LABEL_SMOOTHING = 0.1
IMG_SIZE, CACHE_SIDE = 224, 256
CLIP_NAME = 'openai/clip-vit-base-patch16'
NUM_WORKERS = min(8, os.cpu_count() or 2) if IN_COLAB else 0
SEED = 0

if SMOKE:
    for h in HP.values():
        h.update(epochs=1, batch=16)
    ADS.update(epochs=1, patience=1, seeds=(0,))

# Precisión mixta: bf16 en GPUs que la soportan (L4, A100), fp16 + GradScaler en T4
AMP_DTYPE = torch.bfloat16 if DEVICE == 'cuda' and torch.cuda.is_bf16_supported() else torch.float16
def autocast():
    return torch.autocast('cuda', dtype=AMP_DTYPE) if DEVICE == 'cuda' else contextlib.nullcontext()

def ruta(p):
    return p.replace(DRIVE_ROOT, 'publicity_analysis (Drive)').replace(os.path.expanduser('~'), '~')

print('Resultados en:', ruta(OUT), '· SMOKE =', SMOKE, '· AMP:', AMP_DTYPE if DEVICE == 'cuda' else 'no')
print('Lotes:', {k: v['batch'] for k, v in HP.items()}, '· workers:', NUM_WORKERS)

## 3. EmoSet: índice, partición oficial e imágenes reducidas

Las imágenes originales miden en promedio ~820×730 px. Decodificarlas en cada época es el cuello de botella en Colab (pocos núcleos de CPU). Por eso se crea **una sola vez** una copia con el lado menor de 256 px en el disco de la VM, leyendo directamente desde el zip, y se guarda también como `EmoSet-118K_256.zip` en Drive (~2,5 GB) para las sesiones siguientes.

In [ ]:
Z = zipfile.ZipFile(EMOSET_ZIP)
info = json.loads(Z.read('info.json'))
LABEL2IDX = info['label2idx']
EMOTIONS = [info['idx2label'][str(i)] for i in range(len(LABEL2IDX))]
POS_IDX = [LABEL2IDX[e] for e in ('amusement', 'awe', 'contentment', 'excitement')]
print('Emociones:', EMOTIONS, '· positivas:', POS_IDX)

if os.path.exists(EMOSET_INDEX):
    emo = pd.read_csv(EMOSET_INDEX, usecols=['split', 'emotion', 'image', 'label'])
else:   # mismo índice que crea el notebook 08, leído desde el zip
    rows = [{'split': s, 'emotion': e, 'image': img} for s in ('train', 'val', 'test')
            for e, img, _ in json.loads(Z.read(f'{s}.json'))]
    emo = pd.DataFrame(rows)
    emo['label'] = emo['emotion'].map(LABEL2IDX)

if SMOKE:   # muestra estratificada pequeña
    n = {'train': 32, 'val': 16, 'test': 16}
    emo = pd.concat([g.sample(n[s], random_state=SEED) for (s, _), g in emo.groupby(['split', 'emotion'])],
                    ignore_index=True)

emo['path'] = emo['image'].map(lambda p: os.path.join(IMG_DIR, p))
E = {s: emo[emo['split'] == s].reset_index(drop=True) for s in ('train', 'val', 'test')}
print({s: len(d) for s, d in E.items()})

In [ ]:
def _resize_chunk(args):
    # Cada proceso abre su propio ZipFile (no se puede compartir entre procesos)
    zip_path, rels, out_dir, side = args
    z, n = zipfile.ZipFile(zip_path), 0
    for rel in rels:
        dst = os.path.join(out_dir, rel)
        if os.path.exists(dst):
            continue
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        im = Image.open(io.BytesIO(z.read(rel)))
        im.draft('RGB', (side, side))          # el decodificador JPEG reduce al leer: mucho más rápido
        im = im.convert('RGB')
        w, h = im.size
        s = side / min(w, h)
        if s < 1:
            im = im.resize((round(w * s), round(h * s)), Image.BICUBIC)
        im.save(dst, quality=90)
        n += 1
    return n


def prepare_images(rels):
    faltan = [r for r in rels if not os.path.exists(os.path.join(IMG_DIR, r))]
    if not faltan:
        print(f'{len(rels)} imágenes ya preparadas en {ruta(IMG_DIR)}')
        return
    t0 = time.time()
    if not SMOKE and os.path.exists(CACHE_ZIP):
        print('Copiando imágenes reducidas desde Drive...')
        local = os.path.join(WORK, 'cache_256.zip')
        shutil.copy(CACHE_ZIP, local)
        with zipfile.ZipFile(local) as z:
            z.extractall(IMG_DIR)
        os.remove(local)
        print(f'  listo en {time.time() - t0:.0f}s')
        return
    n_proc = max(1, (os.cpu_count() or 2))
    chunks = [faltan[i::n_proc * 8] for i in range(n_proc * 8)]
    print(f'Reduciendo {len(faltan)} imágenes a {CACHE_SIDE} px con {n_proc} procesos...')
    with mp.get_context('fork').Pool(n_proc) as pool:
        hechos = 0
        for k, n in enumerate(pool.imap_unordered(_resize_chunk, [(EMOSET_ZIP, c, IMG_DIR, CACHE_SIDE) for c in chunks]), 1):
            hechos += n
            if k % max(1, len(chunks) // 10) == 0:
                print(f'  {hechos}/{len(faltan)} ({time.time() - t0:.0f}s)', flush=True)
    print(f'  listo en {time.time() - t0:.0f}s')
    if not SMOKE:   # se guarda en Drive para no repetir este paso en otras sesiones
        local = os.path.join(WORK, 'cache_256.zip')
        with zipfile.ZipFile(local, 'w', zipfile.ZIP_STORED) as z:
            for r in emo['image']:
                z.write(os.path.join(IMG_DIR, r), r)
        shutil.copy(local, CACHE_ZIP)
        os.remove(local)
        print(f'  copia guardada en {ruta(CACHE_ZIP)} ({os.path.getsize(CACHE_ZIP) / 1024**3:.1f} GB)')


prepare_images(emo['image'].tolist())

## 4. Modelos, transformaciones y entrenamiento

- **ResNet-50:** pesos ImageNet de torchvision (los mismos que `pipeline.py`); la capa final se reemplaza por una lineal.
- **CLIP ViT-B/16:** codificador de imagen de OpenAI (`transformers`); sobre su embedding de 512 dimensiones va una capa lineal.
- **Aumentación en EmoSet:** recorte aleatorio y espejo horizontal. **Sin cambios de color ni brillo**: el brillo y el colorido son señales emocionales en EmoSet (notebook 08), alterarlos confundiría al modelo.
- **Anuncios:** la imagen completa se lleva a 224×224 sin recortar bordes (`keep_full_image` del pipeline: no se pierden textos ni logos).
- Optimizador AdamW con tasa menor para el backbone que para la capa nueva, *warm-up* + coseno, *label smoothing* 0,1 y precisión mixta.

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
CLIP_MEAN, CLIP_STD = [0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711]


class Net(nn.Module):
    def __init__(self, kind, n_out):
        super().__init__()
        self.kind = kind
        if kind == 'resnet50':
            m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
            m.fc = nn.Identity()
            self.backbone, self.dim, self.mean, self.std = m, 2048, IMAGENET_MEAN, IMAGENET_STD
        elif kind == 'clip':
            from transformers import CLIPVisionModelWithProjection
            self.backbone = CLIPVisionModelWithProjection.from_pretrained(CLIP_NAME)
            self.dim, self.mean, self.std = self.backbone.config.projection_dim, CLIP_MEAN, CLIP_STD
        else:
            raise ValueError(kind)
        self.head = nn.Linear(self.dim, n_out)

    def embed(self, x):
        return self.backbone(x) if self.kind == 'resnet50' else self.backbone(pixel_values=x).image_embeds

    def forward(self, x):
        return self.head(self.embed(x))


def make_net(kind, n_out, emoset_ckpt=None):
    # emoset_ckpt: pesos de la etapa 1 (8 emociones); la capa final se reemplaza por una nueva
    net = Net(kind, 8 if emoset_ckpt else n_out)
    if emoset_ckpt:
        net.load_state_dict(torch.load(emoset_ckpt, map_location='cpu'))
        net.head = nn.Linear(net.dim, n_out)
    return net.to(DEVICE)


def tf_emoset(train, mean, std):
    ops = [T.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0)), T.RandomHorizontalFlip()] if train else \
          [T.Resize(CACHE_SIDE), T.CenterCrop(IMG_SIZE)]
    return T.Compose(ops + [T.ToTensor(), T.Normalize(mean, std)])


def tf_ads(train, mean, std):
    ops = [T.Resize((IMG_SIZE, IMG_SIZE))]
    if train:
        ops += [T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1)), T.RandomHorizontalFlip()]
    return T.Compose(ops + [T.ToTensor(), T.Normalize(mean, std)])


class ImgDS(Dataset):
    def __init__(self, paths, labels, tf):
        self.paths, self.labels, self.tf = list(paths), list(labels), tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert('RGB')), int(self.labels[i])


def loader(paths, labels, tf, batch, train):
    return DataLoader(ImgDS(paths, labels, tf), batch_size=batch, shuffle=train, num_workers=NUM_WORKERS,
                      pin_memory=DEVICE == 'cuda', drop_last=train and len(paths) > batch)


def make_optim(net, lr_backbone, lr_head, wd, total_steps):
    opt = torch.optim.AdamW([{'params': net.backbone.parameters(), 'lr': lr_backbone},
                             {'params': net.head.parameters(), 'lr': lr_head}], weight_decay=wd)
    warm = max(1, int(0.05 * total_steps))
    f = lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total_steps - warm)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, f)
    scaler = torch.amp.GradScaler('cuda') if DEVICE == 'cuda' and AMP_DTYPE == torch.float16 else None
    return opt, sched, scaler


def train_epoch(net, dl, opt, sched, scaler, loss_fn):
    net.train()
    tot, n = 0.0, 0
    for x, y in dl:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with autocast():
            loss = loss_fn(net(x).float(), y)
        opt.zero_grad(set_to_none=True)
        if scaler:
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
        sched.step()
        tot, n = tot + loss.item() * len(y), n + len(y)
    return tot / n


@torch.no_grad()
def predict_logits(net, dl):
    net.eval()
    ys, out = [], []
    for x, y in dl:
        with autocast():
            out.append(net(x.to(DEVICE, non_blocking=True)).float().cpu())
        ys.append(y)
    return torch.cat(ys).numpy(), torch.cat(out)


@torch.no_grad()
def embeddings(net, paths, tf, batch=256):
    net.eval()
    out = []
    for x, _ in loader(paths, np.zeros(len(paths), int), tf, batch, False):
        with autocast():
            out.append(net.embed(x.to(DEVICE, non_blocking=True)).float().cpu())
    return torch.cat(out).numpy()


def liberar(*objs):
    for o in objs:
        del o
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

## 5. Etapa 1 · Referencias sin ajustar: CLIP *zero-shot* y sondas lineales

- **CLIP *zero-shot*:** cada emoción se describe con frases ("a photo that evokes awe.", "an image that makes people feel awe.", …) y la imagen se asigna a la emoción cuyo texto es más parecido. No usa ninguna etiqueta de EmoSet.
- **Sonda lineal:** el backbone queda congelado; con sus embeddings se entrena solo una capa lineal (en GPU, con *weight decay* elegido en validación).

Los embeddings se guardan en Drive y se reutilizan.

In [ ]:
def emb_cached(name, kind, emoset_ckpt=None):
    path = os.path.join(OUT, f'emb_emoset_{name}.npz')
    if os.path.exists(path):
        d = np.load(path)
        if len(d['train']) == len(E['train']) and len(d['test']) == len(E['test']):
            return {k: d[k].astype(np.float32) for k in ('train', 'val', 'test')}
    net = make_net(kind, 8, emoset_ckpt)
    tf = tf_emoset(False, net.mean, net.std)
    t0 = time.time()
    d = {s: embeddings(net, E[s]['path'], tf) for s in ('train', 'val', 'test')}
    np.savez(path, **{k: v.astype(np.float16) for k, v in d.items()})
    print(f'{name}: embeddings en {time.time() - t0:.0f}s')
    liberar(net)
    return d


def linear_probe(Xtr, ytr, Xva, yva, n_cls, wds=(1e-4, 1e-3, 1e-2), epochs=30):
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
    to_t = lambda X: torch.tensor((X - mu) / sd, dtype=torch.float32, device=DEVICE)
    Xtr_t, Xva_t = to_t(Xtr), to_t(Xva)
    ytr_t = torch.tensor(ytr, device=DEVICE)
    best = (-1, None, None)
    for wd in wds:
        torch.manual_seed(SEED)
        lin = nn.Linear(Xtr.shape[1], n_cls).to(DEVICE)
        opt = torch.optim.AdamW(lin.parameters(), lr=1e-3, weight_decay=wd)
        for _ in range(epochs):
            for idx in torch.randperm(len(Xtr_t), device=DEVICE).split(4096):
                loss = F.cross_entropy(lin(Xtr_t[idx]), ytr_t[idx])
                opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            acc = (lin(Xva_t).argmax(1).cpu().numpy() == yva).mean()
        if acc > best[0]:
            best = (acc, wd, lin)
    lin = best[2]
    return (lambda X: lin(to_t(X)).detach().cpu()), best[1]


def emoset_metrics(name, y, logits):
    pred = logits.argmax(1).numpy()
    return {'modelo': name, 'acc_8': accuracy_score(y, pred), 'macro_f1_8': f1_score(y, pred, average='macro'),
            'acc_valencia_2': (np.isin(y, POS_IDX) == np.isin(pred, POS_IDX)).mean()}


RES_EMOSET, PRED_EMOSET = [], {}
y_tr, y_va, y_te = (E[s]['label'].to_numpy() for s in ('train', 'val', 'test'))

for name, kind in [('ResNet-50 congelada + sonda', 'resnet50'), ('CLIP congelado + sonda', 'clip')]:
    X = emb_cached(kind, kind)
    f, wd = linear_probe(X['train'], y_tr, X['val'], y_va, 8)
    logits = f(X['test'])
    RES_EMOSET.append(emoset_metrics(name, y_te, logits)); PRED_EMOSET[name] = logits
    print(f'{name}: weight decay {wd} · acc test {RES_EMOSET[-1]["acc_8"]:.3f}')
    if kind == 'clip':
        X_clip = X

In [ ]:
# CLIP zero-shot: prompts por emoción, promediados
from transformers import CLIPTextModelWithProjection, CLIPTokenizer

TEMPLATES = ['a photo that evokes {}.', 'an image that makes people feel {}.',
             'a photo expressing {}.', 'a picture that conveys a feeling of {}.']
tok = CLIPTokenizer.from_pretrained(CLIP_NAME)
txt = CLIPTextModelWithProjection.from_pretrained(CLIP_NAME).to(DEVICE).eval()


@torch.no_grad()
def text_classifier(class_words):
    W = []
    for w in class_words:
        t = tok([tp.format(w) for tp in TEMPLATES], padding=True, return_tensors='pt').to(DEVICE)
        z = F.normalize(txt(**t).text_embeds.float(), dim=-1).mean(0)
        W.append(F.normalize(z, dim=0))
    return torch.stack(W).cpu()


W_emo = text_classifier(EMOTIONS)
logits = 100 * F.normalize(torch.tensor(X_clip['test']), dim=-1) @ W_emo.T
RES_EMOSET.append(emoset_metrics('CLIP zero-shot', y_te, logits)); PRED_EMOSET['CLIP zero-shot'] = logits
pd.DataFrame(RES_EMOSET).set_index('modelo')

## 6. Etapa 1 · Ajuste completo en EmoSet

Se guarda en Drive el mejor modelo según la exactitud en validación (`*_best.pt`) y, al final de cada época, un punto de control para reanudar (`*_last.pt`). Si `*_done.json` existe, el modelo ya está entrenado y solo se carga.

In [ ]:
def train_emoset(name, kind):
    hp = HP[kind]
    best_path, last_path, done_path = (os.path.join(OUT, f'{name}_{s}') for s in ('best.pt', 'last.pt', 'done.json'))
    net = Net(kind, 8).to(DEVICE)
    if os.path.exists(done_path):
        net.load_state_dict(torch.load(best_path, map_location=DEVICE))
        print(f'{name}: ya entrenado, se carga {ruta(best_path)}')
        return net, json.load(open(done_path))['hist']
    if kind == 'resnet50' and DEVICE == 'cuda':
        net = net.to(memory_format=torch.channels_last)
    dl_tr = loader(E['train']['path'], E['train']['label'], tf_emoset(True, net.mean, net.std), hp['batch'], True)
    dl_va = loader(E['val']['path'], E['val']['label'], tf_emoset(False, net.mean, net.std), hp['batch'] * 2, False)
    opt, sched, scaler = make_optim(net, hp['lr_backbone'], hp['lr_head'], hp['wd'], hp['epochs'] * len(dl_tr))
    loss_fn = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    start, best, hist = 0, -1.0, []
    if os.path.exists(last_path):
        ck = torch.load(last_path, map_location=DEVICE, weights_only=False)
        net.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
        if scaler and ck['scaler']:
            scaler.load_state_dict(ck['scaler'])
        start, best, hist = ck['epoch'] + 1, ck['best'], ck['hist']
        print(f'{name}: se reanuda desde la época {start}')
    for ep in range(start, hp['epochs']):
        t0 = time.time()
        tl = train_epoch(net, dl_tr, opt, sched, scaler, loss_fn)
        yv, lv = predict_logits(net, dl_va)
        acc = (lv.argmax(1).numpy() == yv).mean()
        hist.append({'epoch': ep, 'train_loss': tl, 'val_acc': acc, 'min': (time.time() - t0) / 60})
        if acc > best:
            best = acc
            torch.save(net.state_dict(), best_path)
        torch.save({'model': net.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(),
                    'scaler': scaler.state_dict() if scaler else None, 'epoch': ep, 'best': best, 'hist': hist}, last_path)
        print(f'{name} · época {ep + 1}/{hp["epochs"]} · pérdida {tl:.4f} · val acc {acc:.4f} · {hist[-1]["min"]:.1f} min', flush=True)
    net.load_state_dict(torch.load(best_path, map_location=DEVICE))
    json.dump({'hist': hist, 'best_val_acc': float(best)}, open(done_path, 'w'), default=float)
    os.remove(last_path)
    return net, hist


CKPT = {}
for name, kind, label in [('resnet50_emoset', 'resnet50', 'ResNet-50 ajustada'), ('clip_emoset', 'clip', 'CLIP ajustado')]:
    net, hist = train_emoset(name, kind)
    CKPT[kind] = os.path.join(OUT, f'{name}_best.pt')
    dl_te = loader(E['test']['path'], E['test']['label'], tf_emoset(False, net.mean, net.std), HP[kind]['batch'] * 2, False)
    y, logits = predict_logits(net, dl_te)
    RES_EMOSET.append(emoset_metrics(label, y, logits)); PRED_EMOSET[label] = logits
    pd.DataFrame(hist).to_csv(os.path.join(OUT, f'{name}_historial.csv'), index=False)
    liberar(net)

In [ ]:
res_emoset = pd.DataFrame(RES_EMOSET).set_index('modelo')
res_emoset['referencia_paper_acc_8'] = pd.Series({'ResNet-50 ajustada': 0.7404})
res_emoset['referencia_paper_valencia_2'] = pd.Series({'ResNet-50 ajustada': 0.9348})
res_emoset.to_csv(os.path.join(OUT, 'etapa1_emoset_resultados.csv'))
display(res_emoset)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, name in zip(axes, ['ResNet-50 ajustada', 'CLIP ajustado']):
    cm = confusion_matrix(y_te, PRED_EMOSET[name].argmax(1).numpy(), labels=range(8))
    cmn = cm / cm.sum(1, keepdims=True)
    ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
    for i in range(8):
        for j in range(8):
            ax.text(j, i, f'{cmn[i, j]:.2f}', ha='center', va='center', fontsize=7,
                    color='white' if cmn[i, j] > 0.5 else 'black')
    ax.set_xticks(range(8), EMOTIONS, rotation=45, ha='right'); ax.set_yticks(range(8), EMOTIONS)
    ax.set(title=f'{name} · test EmoSet', xlabel='predicha', ylabel='real')
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'etapa1_confusion.png'), dpi=150); plt.show()

## 7. Etapa 2 · Anuncios en 3 clases (Positiva · Neutra · Negativa)

Datos y partición de `pipeline.py`: las 2.000 imágenes originales de `images_or/`, etiquetas de `etiquetas_originales.csv` (Felicidad, Diversión, Curiosidad → Positiva; Indiferencia → Neutra; Tristeza, Miedo, Desagrado → Negativa) y **test fijo** (`test_seed = 2024`), el mismo del notebook 01.

### 7a. Transferencia directa (sin entrenar con anuncios)

Los modelos de EmoSet solo conocen positivo y negativo: la probabilidad de valencia positiva es la suma de las 4 emociones positivas. Se evalúa en los anuncios **no neutros** y se mira cómo se reparten los **neutros**.

In [ ]:
cfg_ads = pl.Config(originals_dir='images_or', labels_csv='etiquetas_originales.csv', output_dir=OUT)
ads = pl.load_labels(cfg_ads)
if SMOKE:
    ads = ads.groupby('label', group_keys=False).sample(n=40, random_state=SEED).reset_index(drop=True)
print(f'{len(ads)} anuncios ·', ads['label'].map(dict(enumerate(pl.CLASS_NAMES))).value_counts().to_dict())

transfer_rows, p_pos = [], {}
for kind, label in [('resnet50', 'ResNet-50 · EmoSet'), ('clip', 'CLIP · EmoSet')]:
    net = Net(kind, 8).to(DEVICE)                     # modelo de la etapa 1, con sus 8 salidas
    net.load_state_dict(torch.load(CKPT[kind], map_location=DEVICE))
    y, logits = predict_logits(net, loader(ads['path'], ads['label'], tf_ads(False, net.mean, net.std), 64, False))
    p = F.softmax(logits, 1)[:, POS_IDX].sum(1).numpy()
    p_pos[label] = p
    m = y != 1
    pred = np.where(p[m] >= 0.5, 0, 2)
    transfer_rows.append({'modelo': label, 'anuncios_no_neutros': int(m.sum()),
                          'accuracy': accuracy_score(y[m], pred), 'balanced_accuracy': balanced_accuracy_score(y[m], pred)})
    liberar(net)

# CLIP zero-shot en 3 clases, como referencia sin entrenamiento
net = Net('clip', 3).to(DEVICE)
Xa = embeddings(net, ads['path'], tf_ads(False, net.mean, net.std))
liberar(net)
W_val = text_classifier(['a positive emotion', 'a neutral, indifferent feeling', 'a negative emotion'])
pred_zs = (F.normalize(torch.tensor(Xa), dim=-1) @ W_val.T).argmax(1).numpy()
m_zs = pl.compute_metrics(ads['label'].to_numpy(), pred_zs)
print(f"CLIP zero-shot (3 clases, todos los anuncios): accuracy {m_zs['accuracy']:.3f} · macro-F1 {m_zs['macro_f1']:.3f}")
display(pd.DataFrame(transfer_rows).set_index('modelo'))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
for ax, (label, p) in zip(axes, p_pos.items()):
    for c, col in zip(range(3), ['#4a78b5', '#9a9a9a', '#c0504d']):
        ax.hist(p[ads['label'].to_numpy() == c], bins=20, range=(0, 1), alpha=0.6, density=True,
                label=pl.CLASS_NAMES[c], color=col)
    ax.set(title=label, xlabel='P(valencia positiva) según EmoSet')
axes[0].set_ylabel('densidad'); axes[0].legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'etapa2_transferencia_directa.png'), dpi=150); plt.show()

### 7b. Sonda lineal (backbone congelado)

Regresión logística con pesos de clase balanceados; `C` se elige con validación cruzada estratificada (5 pliegues) sobre train+val, maximizando macro-F1. El test fijo se evalúa una vez, con IC95% por *bootstrap*.

In [ ]:
VARIANTES = [('ResNet-50 · ImageNet', 'resnet50', None), ('ResNet-50 · EmoSet', 'resnet50', CKPT['resnet50']),
             ('CLIP · original', 'clip', None), ('CLIP · EmoSet', 'clip', CKPT['clip'])]
split0 = pl.make_splits(ads, cfg_ads, seed=0)
test_imgs = set(split0['test']['image'])
is_test = ads['image'].isin(test_imgs).to_numpy()
y_ads = ads['label'].to_numpy()


def bootstrap_ci(y, pred, metric, n=1000, seed=0):
    g = np.random.default_rng(seed)
    vals = [metric(y[i], pred[i]) for i in (g.integers(0, len(y), len(y)) for _ in range(n))]
    return np.percentile(vals, [2.5, 97.5])


probe_rows, cms = [], {}
for label, kind, ck in VARIANTES:
    net = make_net(kind, 3, ck)
    X = embeddings(net, ads['path'], tf_ads(False, net.mean, net.std))
    liberar(net)
    Xtr, ytr, Xte, yte = X[~is_test], y_ads[~is_test], X[is_test], y_ads[is_test]
    folds = StratifiedKFold(n_splits=2 if SMOKE else 5, shuffle=True, random_state=SEED)
    best = (-1, None)
    for C in (1e-3, 1e-2, 1e-1, 1.0):
        f1s = []
        for a, b in folds.split(Xtr, ytr):
            sc = StandardScaler().fit(Xtr[a])
            clf = LogisticRegression(C=C, class_weight='balanced', max_iter=5000).fit(sc.transform(Xtr[a]), ytr[a])
            f1s.append(f1_score(ytr[b], clf.predict(sc.transform(Xtr[b])), average='macro'))
        if np.mean(f1s) > best[0]:
            best = (np.mean(f1s), C)
    sc = StandardScaler().fit(Xtr)
    clf = LogisticRegression(C=best[1], class_weight='balanced', max_iter=5000).fit(sc.transform(Xtr), ytr)
    pred = clf.predict(sc.transform(Xte))
    m = pl.compute_metrics(yte, pred)
    lo, hi = bootstrap_ci(yte, pred, lambda a, b: f1_score(a, b, average='macro', labels=[0, 1, 2], zero_division=0))
    probe_rows.append({'variante': label, 'protocolo': 'sonda lineal', 'C': best[1], **{k: m[k] for k in (
        'accuracy', 'balanced_accuracy', 'macro_f1', 'f1_Positive', 'f1_Neutral', 'f1_Negative')},
        'macro_f1_ic95': f'[{lo:.3f}, {hi:.3f}]'})
    cms[(label, 'sonda lineal')] = np.array(m['confusion_matrix'])
    print(f"{label}: C={best[1]} · macro-F1 test {m['macro_f1']:.3f} [{lo:.3f}, {hi:.3f}]")
res_probe = pd.DataFrame(probe_rows)

### 7c. Ajuste completo con 5 semillas

Para cada semilla: partición train/val de `pipeline.make_splits` (el test no cambia), pérdida con pesos de clase (`pipeline.class_weights`), selección de la mejor época por macro-F1 en validación y *early stopping*. Se reporta media e IC95% sobre semillas, como en el notebook 01.

In [ ]:
def finetune_ads(kind, ck, seed):
    pl.set_seed(seed)
    sp = pl.make_splits(ads, cfg_ads, seed)
    net = make_net(kind, 3, ck)
    dl_tr = loader(sp['train']['path'], sp['train']['label'], tf_ads(True, net.mean, net.std), ADS['batch'], True)
    dl_va = loader(sp['val']['path'], sp['val']['label'], tf_ads(False, net.mean, net.std), 64, False)
    dl_te = loader(sp['test']['path'], sp['test']['label'], tf_ads(False, net.mean, net.std), 64, False)
    opt, sched, scaler = make_optim(net, ADS['lr_backbone'][kind], ADS['lr_head'], HP[kind]['wd'], ADS['epochs'] * len(dl_tr))
    loss_fn = nn.CrossEntropyLoss(weight=pl.class_weights(sp['train']).to(DEVICE))
    best, best_ep, state = -1, -1, None
    for ep in range(ADS['epochs']):
        train_epoch(net, dl_tr, opt, sched, scaler, loss_fn)
        yv, lv = predict_logits(net, dl_va)
        f1 = f1_score(yv, lv.argmax(1).numpy(), average='macro', labels=[0, 1, 2], zero_division=0)
        if f1 > best:
            best, best_ep, state = f1, ep, {k: v.detach().clone() for k, v in net.state_dict().items()}
        if ep - best_ep >= ADS['patience']:
            break
    net.load_state_dict(state)
    yt, lt = predict_logits(net, dl_te)
    liberar(net)
    return pl.compute_metrics(yt, lt.argmax(1).numpy()), best_ep


ft_path = os.path.join(OUT, 'etapa2_ajuste_por_semilla.csv')
ft = pd.read_csv(ft_path).to_dict('records') if os.path.exists(ft_path) else []
hechos = {(r['variante'], r['seed']) for r in ft}
for label, kind, ck in VARIANTES:
    for seed in ADS['seeds']:
        if (label, seed) in hechos:
            continue
        t0 = time.time()
        m, ep = finetune_ads(kind, ck, seed)
        ft.append({'variante': label, 'seed': seed, 'best_epoch': ep, 'confusion_matrix': json.dumps(m.pop('confusion_matrix')), **m})
        pd.DataFrame(ft).to_csv(ft_path, index=False)      # se guarda tras cada semilla (reanudable)
        print(f'{label} · semilla {seed}: macro-F1 {m["macro_f1"]:.3f} · mejor época {ep} · {time.time() - t0:.0f}s', flush=True)
ft = pd.DataFrame(ft)

## 8. Resumen de la etapa 2

In [ ]:
METRICS = ['accuracy', 'balanced_accuracy', 'macro_f1', 'f1_Positive', 'f1_Neutral', 'f1_Negative']
rows = []
for label, g in ft.groupby('variante', sort=False):
    row = {'variante': label, 'protocolo': f'ajuste completo ({len(g)} semillas)'}
    for k in METRICS:
        mean, sd, lo, hi = pl.mean_ci(g[k])
        row[k] = mean
        if k == 'macro_f1':
            row['macro_f1_ic95'] = f'[{lo:.3f}, {hi:.3f}]'
    rows.append(row)
    cms[(label, 'ajuste completo')] = sum(np.array(json.loads(c)) for c in g['confusion_matrix'])
resumen = pd.concat([res_probe.drop(columns='C'), pd.DataFrame(rows)], ignore_index=True)
resumen.to_csv(os.path.join(OUT, 'etapa2_anuncios_resumen.csv'), index=False)
display(resumen.set_index(['protocolo', 'variante']))

fig, ax = plt.subplots(figsize=(10, 4))
orden = [v[0] for v in VARIANTES]
for k, (prot, col) in enumerate([('sonda lineal', '#9db7da'), ('ajuste completo', '#4a78b5')]):
    sub = resumen[resumen['protocolo'].str.startswith(prot)].set_index('variante').reindex(orden)
    ci = sub['macro_f1_ic95'].str.strip('[]').str.split(', ', expand=True).astype(float)
    x = np.arange(len(orden)) + (k - 0.5) * 0.38
    ax.bar(x, sub['macro_f1'], width=0.38, color=col, label=prot)
    ax.errorbar(x, sub['macro_f1'], yerr=[sub['macro_f1'] - ci[0], ci[1] - sub['macro_f1']], fmt='none', ecolor='black', capsize=3)
ax.axhline(m_zs['macro_f1'], color='grey', ls=':', label='CLIP zero-shot')
ax.set_xticks(range(len(orden)), orden)
ax.set(ylabel='macro-F1 en test (IC95%)', title='Anuncios · 3 clases: ¿ayuda preentrenar en EmoSet?')
ax.legend(loc='lower right')
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'etapa2_macro_f1.png'), dpi=150); plt.show()

fig, axes = plt.subplots(2, 4, figsize=(16, 7.5))
for r, prot in enumerate(['sonda lineal', 'ajuste completo']):
    for c, label in enumerate(orden):
        cm = cms[(label, prot)]
        cmn = cm / cm.sum(1, keepdims=True)
        ax = axes[r, c]
        ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
        for i in range(3):
            for j in range(3):
                ax.text(j, i, f'{cmn[i, j]:.2f}', ha='center', va='center', color='white' if cmn[i, j] > 0.5 else 'black')
        ax.set_xticks(range(3), pl.CLASS_NAMES); ax.set_yticks(range(3), pl.CLASS_NAMES)
        ax.set_title(f'{label}\n{prot}', fontsize=9)
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'etapa2_confusion.png'), dpi=150); plt.show()

### Cómo leer los resultados

- **Etapa 1:** la fila *ResNet-50 ajustada* debería acercarse al 74 % del artículo de EmoSet (8 clases) y al 93 % en valencia. Si queda muy por debajo, revise el historial (`resnet50_emoset_historial.csv`).
- **Transferencia directa (7a):** si los anuncios neutros se concentran cerca de P = 0,5 en el histograma, el modelo de EmoSet "duda" en ellos, lo que sugiere que la neutralidad se parece a la ambigüedad entre valencias. Si se mezclan con los positivos o negativos, la clase Neutra es otra cosa y solo puede aprenderse de los anuncios.
- **Etapa 2:** la pregunta es si **EmoSet > ImageNet** (o CLIP · EmoSet > CLIP · original) en macro-F1, con IC95% que no se solapen. Mire en especial **F1 de Neutral**, la clase más difícil y la que EmoSet no conoce.